# Pareto Optimization of SEI Additives with ALCHEMI

Lithium-metal and lithium-ion batteries depend on interfaces. 
During cycling, electrolyte molecules touch highly reactive electrode surfaces: the negative electrode can drive reduction chemistry, while the positive electrode can drive oxidation chemistry. 
Some decomposition is harmful because it consumes electrolyte and active lithium, but some early decomposition can be useful when it forms a thin passivating interphase ([Shi et al.](https://www.nature.com/articles/s41524-018-0064-0)).

At the anode, this protective layer is usually called the solid electrolyte interphase (SEI). 
A useful SEI should block electrons, allow Li+ transport, remain chemically and mechanically stable, and avoid continuously attacking the electrolyte ([Shi et al.](https://www.nature.com/articles/s41524-018-0064-0)). 
Electrolyte additives are often chosen because they react before the bulk solvent and help seed a more protective SEI ([Balakrishnan et al.](https://www.sciencedirect.com/science/article/pii/S2451910320300089)). 

A realistic battery-interface simulation would need explicit liquid electrolyte, electron transfer, lithium-ion motion, voltage, many reaction pathways, long timescales, and multiple surface structures. 
The breadth of these modeling challenges is a central theme in the SEI modeling literature ([Shi et al.](https://www.nature.com/articles/s41524-018-0064-0)). 
Instead, this notebook asks you to build a small, transparent screening proxy using the skills from Part 1: generate surface+molecule systems, relax them in ALCHEMI Toolkit batches, compute binding energies, and use those energies to rank candidates.

The two challenge objectives encode the tradeoff --- an additive that barely interacts may do nothing, while one that binds or decomposes too aggressively may create impedance, gas, or unstable products.
First, a molecule should have useful moderate interaction with a reactive Li-metal proxy, which represents SEI seeding. 
Second, once a passivating SEI-like product exists, the molecule should interact weakly with that surface, which represents compatibility with a protective layer. 
Because these goals can conflict, you will use Pareto hypervolume improvement to evaluate candidates.

The chemistry here is intentionally simplified. 
Li metal is a reactive anode proxy. 
Each molecule class maps to one passivating SEI-product proxy surface using the lookup table in `data/class_surface_lookup.csv`. The bundled molecule structures and surface metadata are a starter panel for a workflow exercise, not production battery-interface reference models. 
You should add electrolyte or additive molecules from the literature by providing your own structure and manifest row, and record the citation/provenance for each custom molecule. 
For example, FEC is included because it is a widely studied SEI additive with reported LiF-containing reduction products ([Chen et al.](https://www.pnnl.gov/publications/reduction-mechanism-fluoroethylene-carbonate-stable-solid-electrolyte-interphase-film)).

The reward functions use a moderate-adsorption window for Li-metal seeding and a weak-adsorption preference for SEI passivation, following the same qualitative logic used in SEI-additive and Sabatier-style surface-screening literature ([Lee et al.](https://www.frontiersin.org/journals/energy-research/articles/10.3389/fenrg.2021.654460/full)). 

## What You Need To Produce

Write `outputs/challenge_submission.csv` with one row per molecule you evaluate and these columns:

`candidate_id`, `role`, `molecule_class`, `passivating_surface_id`, `E_bind_Li_eV`, `E_bind_passivating_eV`, `seeding_score`, `passivation_score`, `is_pareto`, `hypervolume_improvement`, `selected`.

Optional but recommended: also write `outputs/raw_component_energies.csv` so the grader can check your binding-energy arithmetic without running any model calls. Additional candidate/provenance columns are allowed; the grader ignores columns outside the required set.


## Control Panel

These defaults follow the cleaned solution pattern: physical slabs, relaxed clean hosts, a compact Part-1-style adsorption grid, and conservative Toolkit relaxation settings. Use `EXAMPLE_SYSTEMS` for smoke tests, then clear it before preparing a full submission.


In [ ]:
from pathlib import Path

TOOLKIT_CHECKPOINT = "medium-mpa-0"
TOOLKIT_HEAD = None
TOOLKIT_DEVICE = "auto"
TOOLKIT_DTYPE = "float32"
TOOLKIT_COMPILE_MODEL = False
TOOLKIT_ENABLE_CUEQ = True
TOOLKIT_DT = 0.005
TOOLKIT_N_STEPS = 5000
TOOLKIT_FMAX = 0.05
TOOLKIT_FIRE2_MAXSTEP = 0.04
TOOLKIT_D3BJ = None
BATCH_SIZE = 2

MIN_ADSORPTION_CLEARANCE_A = 1.6
VDW_HEIGHT_SCALE = 0.66
SURFACE_HEIGHT_TOLERANCE_A = 1.2
GAS_BOX_A = 20.0
ADSORPTION_SITE_LIMIT = 3
ADSORPTION_AZIMUTH_ANGLES_DEG = (0.0,)
MAX_SURFACE_DISPLACEMENT_A = 1.5
FROZEN_SURFACE_FRACTION = 0.5

# For quick debugging you may use, for example:
# EXAMPLE_SYSTEMS = (("FEC", "li_metal"), ("FEC", "passivating"))
# Leave empty for the full challenge panel.
EXAMPLE_SYSTEMS = ()

OUTPUT_DIR = Path("outputs")
SUBMISSION_PATH = OUTPUT_DIR / "challenge_submission.csv"
RAW_COMPONENT_ENERGIES_PATH = OUTPUT_DIR / "raw_component_energies.csv"


## Setup

The [ALCHEMI Toolkit documentation](https://nvidia.github.io/nvalchemi-toolkit/) describes the same core workflow used in Part 1. Keep notebook cells focused on the workflow and put challenge-specific structure/search helpers in a small module under `challenge_utils/`.


In [ ]:
import os
import sys
from importlib.metadata import PackageNotFoundError, version

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "data" / "molecule_manifest.csv").exists():
    candidate = NOTEBOOK_DIR / "challenge-sei"
    if (candidate / "data" / "molecule_manifest.csv").exists():
        NOTEBOOK_DIR = candidate.resolve()
    else:
        raise RuntimeError("Start Jupyter from challenge-sei or from the repository root.")
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

REPO_ROOT = NOTEBOOK_DIR.parent
PART1_ROOT = REPO_ROOT / "part-1-batched-adsorption"
if not (PART1_ROOT / "helpers" / "__init__.py").exists():
    raise RuntimeError("Cannot find Part 1 helpers. Keep challenge-sei beside part-1-batched-adsorption.")
sys.path.insert(0, str(PART1_ROOT))

import numpy as np
import pandas as pd

from helpers import (
    ToolkitRelaxationConfig,
    ToolkitD3BJConfig,
    check_toolkit_native_api,
    get_toolkit_relaxation_engine,
    ase_to_atomic_data,
    atomic_data_to_ase,
    build_slab,
    make_active_mask,
    display_widgets_grid,
)
from helpers.config_search import Configuration
from challenge_utils.pareto import dominates, hypervolume_2d, pareto_flags
from challenge_utils.rewards import passivation_score, seeding_score

print(f"Challenge folder : {NOTEBOOK_DIR.name}")
print(f"Part 1 helpers   : {PART1_ROOT.relative_to(REPO_ROOT)}")
for pkg in ("ase", "numpy", "pandas", "torch", "nvalchemi-toolkit", "ovito"):
    try:
        print(f"{pkg:<18}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:<18}: not installed")


## 1. Load The Challenge Manifests

Load the starter molecules, `data/custom_molecule_manifest.csv` with the candidates you find in literature, surface metadata, and class-to-surface lookup.


In [ ]:
# TODO: Load data/molecule_manifest.csv, data/surface_manifest.csv, and
# data/class_surface_lookup.csv with pandas. If data/custom_molecule_manifest.csv
# exists, append it to molecules_df after checking candidate_id uniqueness.
# Merge molecules_df with the lookup table so each molecule has
# passivating_surface_id.
#
# Then build:
# - challenge_df: all candidates with passivating_surface_id.
# - run_systems_df: one row per adsorption system to calculate.
# - run_challenge_df: molecule-reference rows needed for the active run.
#
# If EXAMPLE_SYSTEMS is non-empty, restrict run_systems_df to those
# (candidate_id, interaction) pairs. Otherwise include both li_metal and
# passivating interactions for every row in challenge_df.

# molecules_df = ...
# surfaces_df = ...
# lookup_df = ...
# challenge_df = ...
# run_systems_df = ...
# run_challenge_df = ...

# assert challenge_df["passivating_surface_id"].notna().all()
# assert {"EC", "EMC"}.issubset(set(challenge_df.loc[challenge_df["role"].eq("baseline"), "candidate_id"]))
# display(challenge_df[["candidate_id", "role", "molecule_class", "passivating_surface_id", "structure_path"]])
# display(surfaces_df[["surface_id", "role", "provenance"]])
# display(run_systems_df[["candidate_id", "interaction", "surface_id", "role", "structure_path"]])

raise NotImplementedError("Load manifests and build the active run tables.")


## 2. Build The Toolkit Relaxation Engine

This is the same native Toolkit path used in Part 1.


In [ ]:
status = check_toolkit_native_api()
print(status["message"])
if not status["available"]:
    raise RuntimeError("ALCHEMI Toolkit native API is not available in this kernel.")

if isinstance(TOOLKIT_D3BJ, dict):
    TOOLKIT_D3BJ = ToolkitD3BJConfig(**TOOLKIT_D3BJ)

relaxation_config = ToolkitRelaxationConfig(
    name="toolkit",
    cache_dir=(OUTPUT_DIR / "cache_json").as_posix(),
    use_cached_responses=False,
    toolkit_checkpoint=TOOLKIT_CHECKPOINT,
    toolkit_head=TOOLKIT_HEAD,
    toolkit_device=TOOLKIT_DEVICE,
    toolkit_dtype=TOOLKIT_DTYPE,
    toolkit_compile_model=TOOLKIT_COMPILE_MODEL,
    toolkit_enable_cueq=TOOLKIT_ENABLE_CUEQ,
    toolkit_dt=TOOLKIT_DT,
    toolkit_n_steps=TOOLKIT_N_STEPS,
    toolkit_fmax=TOOLKIT_FMAX,
    toolkit_fire2_maxstep=TOOLKIT_FIRE2_MAXSTEP,
    toolkit_d3bj=TOOLKIT_D3BJ,
    toolkit_require_d3bj=TOOLKIT_D3BJ is not None,
)
RELAXATION_ENGINE = get_toolkit_relaxation_engine(relaxation_config)
print(f"Toolkit relaxation engine ready: {RELAXATION_ENGINE.name}")


## 3. Structure And Search Helpers

Follow the Part 1 tutorial pattern

- build physical slabs from bulk structures with `build_slab` or ASE surface builders;
- freeze the bottom slab fraction with `make_active_mask(..., bottom_fraction=0.5)`;
- relax clean slabs before building molecule-on-surface starts;
- define explicit named molecular orientations such as `F-down` and `O-down` rather than guessing one contact atom;
- generate a compact `site x orientation x rotation x height` grid, using a scaled van der Waals contact estimate for initial heights;
- validate convergence, frozen atoms, molecule integrity, and surface displacement before selecting the lowest-energy valid start.


In [ ]:
# TODO: Put your implementation in your own helper module, for example
# challenge_utils/my_helpers.py,
# and import the functions here. Keep the notebook workflow concise.
#
# Your helper module should provide functions with responsibilities like:
# - load_atoms(...) and gas_box(...)
# - prepare_challenge_tables(...)
# - build physical Li_metal and passivating slabs from bulk/slab builders
# - surface_active_mask(...), using make_active_mask with bottom_fraction=0.5
# - build_sei_config_grid(...), following Part 1's Configuration pattern
# - make_gas_jobs(...), make_clean_surface_jobs(...), make_combined_jobs(...)
# - relax_structures(...), require_all_converged(...)
# - select_lowest_energy_site_results(...)
# - component_energy_table(...) and binding_energy_table(...)
# - write_ovito_inspection_structures(...)
#
# The key chemistry lesson from FEC/Li is that intact adsorption may prefer
# O-down even though F-down is the LiF-forming precursor. Therefore F-down
# should be sampled explicitly, but true LiF formation would require a separate
# product/dissociation calculation rather than the intact-molecule audit here.

# from challenge_utils.my_helpers import (...)

raise NotImplementedError("Create/import the structure, search, relaxation, and validation helpers.")


## 4. Build And Relax The Jobs

You need the same component energies as Part 1. Relax gas molecules and clean surfaces first, then build the adsorption grid on the relaxed clean slabs.

1. Gas molecules: `E_species`.
2. Clean surfaces: `E_surface` for `Li_metal` and every passivating surface used by the candidates.
3. Combined systems: `E_surface+species` for molecule on Li metal and molecule on its class-specific passivating surface.


In [ ]:
# TODO: Use your helper module to build and relax the three job groups.
#
# Required workflow:
# 1. Load molecule structures for run_challenge_df.
# 2. Build physical adsorption slabs for every surface in run_systems_df.
# 3. Relax gas_jobs and clean_surface_jobs.
# 4. Require gas and clean-surface convergence.
# 5. Build combined_jobs from the relaxed clean-surface structures.
# 6. Relax all site/orientation starts.
# 7. Select the lowest-energy valid start per (candidate_id, interaction).
#
# Valid selected starts should satisfy:
# - Toolkit converged=True and fmax below TOOLKIT_FMAX.
# - frozen atoms stayed fixed.
# - molecule_geometry_flags is empty for intact adsorption scoring.
# - surface_geometry_flags is empty; use MAX_SURFACE_DISPLACEMENT_A as a guard.

# molecule_atoms = ...
# adsorption_surface_atoms = ...
# gas_jobs = ...
# clean_surface_jobs = ...
# gas_results = relax_structures(gas_jobs, RELAXATION_ENGINE, ...)
# clean_surface_results = relax_structures(clean_surface_jobs, RELAXATION_ENGINE, ...)
# require_all_converged(gas_results, label="gas references")
# require_all_converged(clean_surface_results, label="clean-surface references")
# combined_jobs = make_combined_jobs(..., clean_surface_results, ...)
# all_combined_results = relax_structures(combined_jobs, RELAXATION_ENGINE, ...)
# combined_results = select_lowest_energy_site_results(all_combined_results)
# require_all_converged(combined_results, label="selected combined adsorption systems")

raise NotImplementedError("Build and relax the gas, clean-surface, and combined jobs.")


## 5. Compute Binding Energies

Use the same adsorption-energy convention as Part 1: `E_bind = E_surface+species - E_surface - E_species`.


In [ ]:
# TODO: Convert gas_results, clean_surface_results, and combined_results into
# one raw-energy row for Li_metal and one raw-energy row for each candidate's
# passivating surface.
#
# Required columns:
# candidate_id, interaction, surface_id, E_surface_species_eV, E_surface_eV, E_species_eV
# Recommended audit columns:
# selected_site_label, selected_start_orientation, selected_azimuth_deg

# raw_component_energies_df = ...
# raw_component_energies_df.to_csv(RAW_COMPONENT_ENERGIES_PATH, index=False)
# display(raw_component_energies_df.head())

raise NotImplementedError("Build the raw component-energy table.")


In [ ]:
# TODO: Use raw_component_energies_df to compute E_bind_Li_eV and
# E_bind_passivating_eV for every candidate in run_challenge_df.
#
# binding_df should keep:
# candidate_id, role, molecule_class, passivating_surface_id,
# E_bind_Li_eV, E_bind_passivating_eV

# binding_df = ...
# display(binding_df[["candidate_id", "role", "E_bind_Li_eV", "E_bind_passivating_eV"]])

raise NotImplementedError("Compute binding energies from the raw component energies.")


## 5b. Visual Inspect Relaxed Adsorption Geometries With OVITO

Before scoring, inspect the selected relaxed structures. The workflow should choose the lowest-energy valid intact adsorption start.


In [ ]:
# TODO: Write selected combined structures to outputs/ovito_structures as EXTXYZ
# and display them with display_widgets_grid when available.
#
# Recommended helper responsibilities:
# - write_ovito_inspection_structures(combined_results, output_dir=...)
# - choose_inspection_candidates(binding_df)
# - inspection_widget_rows(inspection_df, candidate_ids)

# OVITO_STRUCTURE_DIR = OUTPUT_DIR / "ovito_structures"
# inspection_df = ...
# display_widgets_grid(...)

raise NotImplementedError("Write and inspect selected relaxed structures.")


## 6. Compute Literature-Motivated Reward Scores

The reward functions are provided in `challenge_utils.rewards` so everyone uses the same challenge rubric. They convert each binding energy into an adsorption-strength magnitude `strength = max(0, -E_bind)` then apply bounded rewards.


In [ ]:
# TODO: Apply the provided scalar reward functions to the binding energies.
# scored_df = binding_df.copy()
# scored_df["seeding_score"] = scored_df["E_bind_Li_eV"].map(seeding_score)
# scored_df["passivation_score"] = scored_df["E_bind_passivating_eV"].map(passivation_score)
# display(scored_df[["candidate_id", "role", "seeding_score", "passivation_score"]])

raise NotImplementedError("Compute seeding_score and passivation_score.")


## 7. Pareto Front And Hypervolume Improvement

Treat both scores as objectives to maximize. The baseline front is built from `EC` and `EMC`; additives are evaluated by how much they improve the baseline hypervolume.


In [ ]:
# Pareto helpers are imported from challenge_utils.pareto so you can focus on
# using them correctly.
# TODO: Add is_pareto and hypervolume_improvement columns to scored_df using
# pareto_flags(...) and hypervolume_2d(...).

# final_df = ...
# display(final_df.sort_values("hypervolume_improvement", ascending=False))

raise NotImplementedError("Compute Pareto flags and hypervolume improvements.")


## 8. Select Your Additive And Submit

Mark exactly one additive as `selected=True`: the additive with the maximum hypervolume improvement. Baseline rows should not be selected.


In [ ]:
# TODO: Create a boolean selected column in final_df. Exactly one additive should
# be selected, and baseline rows should never be selected.

# submission = final_df.copy()
# selected_id = ...
# submission["selected"] = submission["candidate_id"].eq(selected_id)
# display(submission[["candidate_id", "role", "hypervolume_improvement", "selected"]])

raise NotImplementedError("Select exactly one additive.")


In [ ]:
required_columns = [
    "candidate_id", "role", "molecule_class", "passivating_surface_id",
    "E_bind_Li_eV", "E_bind_passivating_eV", "seeding_score",
    "passivation_score", "is_pareto", "hypervolume_improvement", "selected",
]
missing = [column for column in required_columns if column not in submission.columns]
if missing:
    raise RuntimeError(f"Submission is missing required columns: {missing}")
if int(submission["selected"].sum()) != 1:
    raise RuntimeError("Exactly one row must be selected.")

OUTPUT_DIR.mkdir(exist_ok=True)
submission[required_columns].to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}")
display(submission[required_columns])


## References And Further Reading

- NVIDIA [ALCHEMI Toolkit documentation](https://nvidia.github.io/nvalchemi-toolkit/) for `AtomicData`, `Batch`, model wrappers, and Toolkit dynamics.
- Batatia et al., [MACE: Higher Order Equivariant Message Passing Neural Networks for Fast and Accurate Force Fields](https://openreview.net/forum?id=YPpSngE-ZU), NeurIPS 2022.
- Larsen et al., [The Atomic Simulation Environment - a Python library for working with atoms](https://doi.org/10.1088/1361-648X/aa680e), J. Phys.: Condens. Matter 2017.
- Stukowski, [Visualization and analysis of atomistic simulation data with OVITO - the Open Visualization Tool](https://doi.org/10.1088/0965-0393/18/1/015012), Modelling Simul. Mater. Sci. Eng. 2010.
- Leung et al., [Stability of Solid Electrolyte Interphase Components on Lithium Metal and Reactive Anode Material Surfaces](https://doi.org/10.1021/acs.jpcc.5b11719), J. Phys. Chem. C 2016; examples use large periodic SEI/Li cells and matching slab-interface references.
- Chanussot et al., [The Open Catalyst 2020 Dataset and Community Challenges](https://doi.org/10.1021/acscatal.0c04525), ACS Catalysis 2021; summarizes common slab-adsorbate setup, adsorption-energy references, vacuum, and fixed subsurface atoms.
- Shi et al., [Review on modeling of the anode solid electrolyte interphase (SEI) for lithium-ion batteries](https://www.nature.com/articles/s41524-018-0064-0), npj Computational Materials 2018.
- Xu et al., [A review on electrolyte additives for lithium-ion batteries](https://www.sciencedirect.com/science/article/pii/S0378775306017538), J. Power Sources 2007.
- Balakrishnan et al., [Electrolyte additives for improved lithium-ion battery performance and overcharge protection](https://www.sciencedirect.com/science/article/pii/S2451910320300089), 2020.
- Li et al., [A Review of Solid Electrolyte Interphases on Lithium Metal Anode](https://pmc.ncbi.nlm.nih.gov/articles/PMC5063117/), Advanced Science 2016.
- [Insights into the efficient roles of solid electrolyte interphase derived from vinylene carbonate additive in rechargeable batteries](https://www.sciencedirect.com/science/article/abs/pii/S1572665722001187), 2022.
- Zhang et al., [Reduction Mechanism of Fluoroethylene Carbonate for Stable Solid-Electrolyte Interphase Film on Silicon Anode](https://www.pnnl.gov/publications/reduction-mechanism-fluoroethylene-carbonate-stable-solid-electrolyte-interphase-film), ChemSusChem 2013.
- Lee et al., [The Sabatier Principle in Electrocatalysis: Basics, Limitations, and Extensions](https://www.frontiersin.org/journals/energy-research/articles/10.3389/fenrg.2021.654460/full), Frontiers in Energy Research 2021.
- Aich et al., [Determination of thermodynamic parameters in adsorption studies: a review](https://link.springer.com/article/10.1007/s11696-025-04218-x), Chemical Papers 2025.
- [Hypervolume bibliography](https://hypervolume.org/bibliography.html) for Pareto hypervolume indicator references.
